### 159. Longest Substring with At Most Two Distinct Characters (Longest Substring with a Subset of the Alphabet)

### Question
Given a string, find the longest substring that contains at most 2 distinct characters and return its length and the index it starts from.

### Example
For example the string "ababcbcbaaabbdef" has four non-trivial (longer than 2 character) sequences:

```
    4                          6             2
----------           -------------------   -----
a, b, a, b, c, b, c, b, a, a,  a,  b,  b,  d,  e,  f
0  1  2  3  4  5  6  7  8  9  10  11  12  13  14  15
         -------------             ---------   -----
               5                       3         2
```


Your solution should return the pair (6, 7) (the length of "baaabb", and the index of the first byte of the sequence in the string).

### Clarifying Questions
- Can the input string be empty?
  - The input string is gauranteed to be nonempty.
- Are only certain characters considered?
  - Any valid character is allowed. Your function should return the length and starting index of the longest substring that satisfies the criteria.
- What if more than one substring has the same maximum length?
  - If multiple substrings match the criteria, return the one with the smallest starting index.

### Followup:
Write a function to do the same thing but with a streaming input.



### Streaming with OrderedDict

- 時間複雜度：$O(n)$
  - 每次操作 OrderedDict 都是 $O(1)$。。
  - 因此，總時間複雜度與字串長度 $n$ 成線性關係。
- 空間複雜度：$O(1)$
  - 使用一個雜湊表 char_map。
  - 因為題目規定字元種類最多只有 2 種（超過就會收縮），所以雜湊表的大小最大為 3。
  - 空間消耗與輸入字串長度無關，屬於常數空間。

left 直接跳轉到目標位置，提升「空間效率與存取限制」（不需要回頭看 s[left]）

In [1]:
from collections import OrderedDict

def sequence_length(s: str):
    # last_seen 紀錄「字元: 最後出現的索引」
    last_seen = OrderedDict() # space: O(2) = O(1)
    left = 0
    max_len, start_idx = 0, 0
    
    for right, char in enumerate(s):  # time: O(n)
        # 1. 更新字元位置：先刪再加，確保它排在 OrderedDict 最後面 (最新)
        if char in last_seen:
            del last_seen[char]
        last_seen[char] = right
        
        # 2. 種類超過 2 時，直接踢掉最久沒出現的
        if len(last_seen) > 2:
            # popitem(last=False) 會移除並回傳「第一筆」(最老) 資料
            _, last_idx = last_seen.popitem(last=False)
            # 左邊界直接「跳」到該字元最後位置的下一格
            left = last_idx + 1
            
        # 3. 更新最大長度
        if right - left + 1 > max_len:
            max_len = right - left + 1
            start_idx = left
            
    return (max_len, start_idx)

In [2]:
# 測試範例
s = "ababcbcbaaabbdef"
result = sequence_length(s)
print(f"結果：{result}") # 預期輸出: (6, 7)

結果：(6, 7)


### Sliding Window

- 時間複雜度：$O(n)$
  - 雖然程式碼中有一個 while 迴圈，但 left 指標與 right 指標在整個過程中都只會從左向右移動一次。
  - 每個字元最多被「移入」視窗一次且「移出」視窗一次。
  - 因此，總時間複雜度與字串長度 $n$ 成線性關係。
- 空間複雜度：$O(1)$
  - 使用一個雜湊表 char_map。
  - 因為題目規定字元種類最多只有 2 種（超過就會收縮），所以雜湊表的大小最大為 3。
  - 空間消耗與輸入字串長度無關，屬於常數空間。
  

面試小撇步：這個解法已經是該問題的最優解法（線性時間）。如果字元種類限制從 2 變成 $k$，此邏輯依然適用，僅需將 len(char_map) > 2 改為 > k。

In [3]:
def sequence_length(s: str):
    # 使用 Hash Map 紀錄字元出現次數
    counts = {} # space: O(2) = O(1)
    left = 0
    max_len, start_idx = 0, 0
    
    for right in range(len(s)): # time: O(n)
        # 1. 移入右邊界字元
        char = s[right]
        counts[char] = counts.get(char, 0) + 1
        
        # 2. 種類超過 2 時，縮減左邊界
        while len(counts) > 2:
            left_char = s[left]
            counts[left_char] -= 1
            if counts[left_char] == 0:
                del counts[left_char]
            left += 1
            
        # 3. 更新最大長度
        if right - left + 1 > max_len:
            max_len = right - left + 1
            start_idx = left
            
    return (max_len, start_idx)

In [4]:
# 測試範例
s = "ababcbcbaaabbdef"
result = sequence_length(s)
print(f"結果：{result}") # 預期輸出: (6, 7)

結果：(6, 7)
